# 1.0 - Exploración Inicial y Validación de Hipótesis de MiBici
**Autor:** David Carrera Castillo (DCC) - A01840136  
**Equipo:** 9 - TC5061  

### Propósito del Notebook
Este notebook documenta la exploración inicial de los datos de MiBici (abril a septiembre de 2026), analiza el comportamiento de las estaciones, verifica la distribución de viajes espurios (< 1 min con mismo O-D) y valida las variables objetivo para las tres preguntas del proyecto:
- **P1:** Demanda horaria por estación.
- **P2:** Riesgo de vaciado/llenado en ventana de 2 horas.
- **P3:** Perfiles de uso y patrones horarios por estación.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from mibici.config import PROCESSED_DATA_DIR, FIGURES_DIR
from mibici.features import (
    cargar_datos_procesados,
    filtrar_viajes_invalidos,
    generar_dataset_p1,
    generar_dataset_p2,
    generar_dataset_p3,
    particionar_temporalmente
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')


## 1. Carga de Datos Procesados
Se cargan las tablas limpias de `viajes.parquet` y `estaciones.parquet` producidas por la ingesta reproducible (`make data`).


In [ ]:
viajes, estaciones = cargar_datos_procesados()
print(f"Total viajes: {len(viajes):,}")
print(f"Total estaciones: {len(estaciones):,}")
viajes.head()


## 2. Detección y Análisis de Viajes Espurios
En los datos existen viajes con duración menor a 60 segundos que inician y terminan en la misma estación.


In [ ]:
duracion_seg = (viajes['fin_del_viaje'] - viajes['inicio_del_viaje']).dt.total_seconds()
mismo_od = (viajes['origen_id'] == viajes['destino_id']) & (duracion_seg < 60)
print(f"Viajes espurios detectados: {mismo_od.sum():,} ({mismo_od.mean()*100:.2f}%)")

viajes_validos = filtrar_viajes_invalidos(viajes)
print(f"Viajes válidos para modelado: {len(viajes_validos):,}")


## 3. Exploración de P1: Demanda Horaria
Distribución de viajes por hora del día y proporción de horas con cero demanda.


In [ ]:
df_p1 = generar_dataset_p1(viajes, estaciones)
print(f"Tamaño tablón P1: {df_p1.shape}")
print(f"Porcentaje de registros con demanda = 0: {(df_p1['demanda'] == 0).mean()*100:.2f}%")

plt.figure(figsize=(10, 4))
df_p1.groupby('hora')['demanda'].mean().plot(kind='bar', color='#1f77b4', edgecolor='black')
plt.title('P1 - Promedio de Salidas por Hora del Día (Todas las Estaciones)')
plt.xlabel('Hora del Día')
plt.ylabel('Promedio de Viajes Iniciados')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 4. Exploración de P2: Riesgo de Rebalanceo en 2 Horas
Distribución del saldo neto futuro (t+1, t+2) y tasa de riesgo según el 30% de la capacidad de la estación.


In [ ]:
df_p2 = generar_dataset_p2(viajes, estaciones)
print(f"Tamaño tablón P2: {df_p2.shape}")
print(f"Tasa global de observaciones en riesgo: {df_p2['riesgo'].mean()*100:.2f}%")

plt.figure(figsize=(8, 4))
sns.histplot(df_p2['saldo_futuro_2h'], bins=50, kde=True, color='#2ca02c')
plt.title('P2 - Distribución del Saldo Neto Futuro en 2 Horas (Llegadas - Salidas)')
plt.xlabel('Saldo Neto Futuro (t+1, t+2)')
plt.ylabel('Frecuencia')
plt.xlim(-30, 30)
plt.tight_layout()
plt.show()


## 5. Exploración de P3: Perfiles de Uso por Estación
Matriz de perfiles de estación basada únicamente en el conjunto de entrenamiento (abril a agosto).


In [ ]:
df_p3 = generar_dataset_p3(viajes, estaciones, solo_train=True)
print(f"Dimensiones de matriz de perfiles P3: {df_p3.shape}")
cols_hora = [c for c in df_p3.columns if c.startswith('prop_hora_')]
print(f"Estaciones analizadas: {len(df_p3)}")

plt.figure(figsize=(10, 4))
plt.scatter(df_p3['longitude'], df_p3['latitude'], c=df_p3['prop_fin_de_semana'], cmap='viridis', s=df_p3['dpcapacity']*2, alpha=0.7)
plt.colorbar(label='Proporción uso en Fin de Semana')
plt.title('P3 - Coordenadas de Estaciones y Uso en Fin de Semana')
plt.xlabel('Longitud')
plt.ylabel('Latitud')
plt.tight_layout()
plt.show()
